# OpenPlaque — LAD Mask-Gate Diagnostic v1
The feasible 46 mm LAD→aorta search still died early. This run instruments the exact prior hard-mask beam, then runs a diagnostic shadow beam with only the hard coronary-mask-proximity rejection removed. The shadow result cannot update accepted anatomy.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DRIVE_ROOT='/content/drive/MyDrive/OpenPlaque'
OUTPUT_DIR=DRIVE_ROOT+'/Left_Main_LAD_Mask_Gate_Diagnostic_v1'
BRANCH='left-main-lad-mask-gate-diagnostic-from-main'
PINNED_SCIENCE_COMMIT='6f0921d8cc22ad9ef0cf627334cb5467f3aa844c'
BASELINE='0593b453959f5a353d644267fbeef24b514ef4d7'
EXPECTED_ALGORITHM='left-main-lad-mask-gate-diagnostic-v1.0'
print('Branch:',BRANCH)
print('Pinned science commit:',PINNED_SCIENCE_COMMIT)
print('Output:',OUTPUT_DIR)


In [ ]:
import os,shutil
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
!git clone --depth 20 --branch $BRANCH https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git -C /content/OpenPlaque checkout --detach $PINNED_SCIENCE_COMMIT
HEAD=(!git -C /content/OpenPlaque rev-parse HEAD)[0].strip()
print('Checked out HEAD:',HEAD)
assert HEAD==PINNED_SCIENCE_COMMIT,(HEAD,PINNED_SCIENCE_COMMIT)
mb=(!git -C /content/OpenPlaque merge-base HEAD $BASELINE)[0].strip()
print('Merge base:',mb)
assert mb==BASELINE


In [ ]:
%pip uninstall -y openplaque >/dev/null 2>&1
%pip install -q --no-cache-dir --force-reinstall --no-deps /content/OpenPlaque
%pip install -q pytest SimpleITK scipy pandas matplotlib numpy pydicom psutil


In [ ]:
import sys,importlib,pathlib,pytest
for name in list(sys.modules):
    if name=='openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
importlib.invalidate_caches()
import openplaque
from openplaque import left_main_lad_mask_gate_diagnostic_v1 as exp
print('openplaque:',openplaque.__file__)
print('experiment:',exp.__file__)
print('algorithm:',exp.ALGORITHM)
assert exp.BASELINE==BASELINE
assert exp.ALGORITHM==EXPECTED_ALGORITHM
compile(pathlib.Path(exp.__file__).read_text(),exp.__file__,'exec')
print('synthetic diagnostic:',exp.synthetic_attrition_self_test())
rc=pytest.main(['-q','/content/OpenPlaque/tests/test_left_main_lad_mask_gate_diagnostic_v1.py'])
if rc!=0: raise RuntimeError(f'pytest failed with exit code {rc}')


In [ ]:
from pathlib import Path
import json
root=Path(DRIVE_ROOT)
required=[root/'Left_Main_Proximal_LAD_Long_Reach_v1_2/summary.json',root/'Left_Coronary_Source_Ostium_Multiseed_Control_Adjudication_v2_1/summary.json',root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',root/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',root/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',root/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries/coronary_arteries.nii.gz',root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries_LEGACY/coronary_arteries.nii.gz',root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/aorta.nii.gz']
missing=[str(p) for p in required if not p.exists()]
print('Preflight required:',len(required),'missing:',len(missing))
if missing: raise FileNotFoundError('\n'.join(missing))
prior=json.loads(required[0].read_text())
print('Prior long-reach status:',prior.get('status'))
print('Prior frontier count:',prior.get('proximal_LAD_long_reach',{}).get('frontier_count'))


In [ ]:
import time,gc
from openplaque.left_main_lad_mask_gate_diagnostic_v1 import run
gc.collect(); t0=time.time()
result=run(DRIVE_ROOT,OUTPUT_DIR)
s=result['summary']
print('ELAPSED MIN:',round((time.time()-t0)/60,2))
print('STATUS:',s.get('status'))
print('RCA CONTROL PASS:',s.get('RCA_control_pass'))
print('LAD START→AORTA MM:',s.get('LAD_start_aorta_distance_mm'))
print('HARD MASK:',s.get('hard_mask_search'))
print('SHADOW SOFT MASK:',s.get('shadow_soft_mask_search'))
print('REPORT:',result.get('report'))
print('ZIP:',result.get('zip'))
